In [125]:
import numpy as np
import pandas as pd

In [126]:
df=pd.read_excel("Online Retail.xlsx")

In [127]:
df.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047.0,United Kingdom


In [128]:
df.shape

(541909, 8)

In [129]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


>**Из этих данных следует что есть пропущенные данные в столбцах Description и CustomerID, что необходимо учесть в дальнейшем анализе.**

In [130]:
df.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386048,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


> **По строке min можно наблюдать что есть отрицательные значения в столбцах Quantity и UnitPrice, которое говорит о возвратах (Quantity<0) и возможных ошибках в указанной цене (UnitPrice<0)**


-----

#### Анализ отрицательных значений в UnitPrice:

In [131]:
df.loc[df["UnitPrice"]<0]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


 >  **По _Description_ видно что платежи с отрицательным _UnitPrice_ не связаны с продажами и не являются ошибочно внесенной ценой.  По _StockCode_	и Description понятно что это бухгалтерские корректировки (долг/списание) и их учет в дальнейшем анализе может исказить реальные выводы. Лучший вариант сохранить эти значения в отдельной таблице и очистить оригинальную.**

In [132]:
df_acc_adjs=df.loc[df["StockCode"]=='B']
df_acc_adjs

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299982,A563185,B,Adjust bad debt,1,2011-08-12 14:50:00,11062.06,NaN,United Kingdom
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


------

#### Проверка логики - "Конкретный заказ один клиент" (1 к 1 соответствие):

In [133]:
df.groupby('InvoiceNo')["CustomerID"].nunique().sort_values(ascending=False)

InvoiceNo
C581569    1
536365     1
536366     1
536367     1
536368     1
          ..
581435     0
581431     0
581498     0
581497     0
581492     0
Name: CustomerID, Length: 25900, dtype: int64

> **Максимальное количество уникальных значений для заказа 1, что значит что один конкретный заказ сделан одним клиентом. 0 отвечает за анонимных клиентов (CustonerID=NaN).**

-------

#### Проверка возвратов и соответствия между Quantity и InvoiceNo:

In [134]:
df['IsReturn']=False
df.loc[df["InvoiceNo"].astype(str).str.lower().str.startswith("c"),'IsReturn']=True

In [135]:
df.loc[df['IsReturn']==True]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsReturn
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom,True
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom,True
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom,True
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom,True
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom,True
...,...,...,...,...,...,...,...,...,...
540449,C581490,23144,ZINC T-LIGHT HOLDER STARS SMALL,-11,2011-12-09 09:57:00,0.83,14397.0,United Kingdom,True
541541,C581499,M,Manual,-1,2011-12-09 10:28:00,224.69,15498.0,United Kingdom,True
541715,C581568,21258,VICTORIAN SEWING BOX LARGE,-5,2011-12-09 11:57:00,10.95,15311.0,United Kingdom,True
541716,C581569,84978,HANGING HEART JAR T-LIGHT HOLDER,-1,2011-12-09 11:58:00,1.25,17315.0,United Kingdom,True


In [136]:
df.loc[df['Quantity']<0,'IsReturn']=True

In [137]:
df.loc[df['IsReturn']==True]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsReturn
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom,True
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom,True
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom,True
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom,True
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom,True
...,...,...,...,...,...,...,...,...,...
540449,C581490,23144,ZINC T-LIGHT HOLDER STARS SMALL,-11,2011-12-09 09:57:00,0.83,14397.0,United Kingdom,True
541541,C581499,M,Manual,-1,2011-12-09 10:28:00,224.69,15498.0,United Kingdom,True
541715,C581568,21258,VICTORIAN SEWING BOX LARGE,-5,2011-12-09 11:57:00,10.95,15311.0,United Kingdom,True
541716,C581569,84978,HANGING HEART JAR T-LIGHT HOLDER,-1,2011-12-09 11:58:00,1.25,17315.0,United Kingdom,True


> **При проверке _InvoiceNo_ на наличие штампа о возврате (условие где номер начинается с буквы с) выявилось 9288 возвратов. При этом проверка на отрицательные значения в столбце _Quantity_ расширили количество флагов с _IsReturn=True_. Из этого следует что есть строки с отрицательным _Quantity_ не отмечеными как возврат.**

In [138]:
df1=df.loc[df["InvoiceNo"].astype(str).str.lower().str.startswith("c")==False]
df1=df1.loc[df1['IsReturn']==True]
df1

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsReturn
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom,True
4347,536764,84952C,NaN,-38,2010-12-02 14:42:00,0.0,NaN,United Kingdom,True
7188,536996,22712,NaN,-20,2010-12-03 15:30:00,0.0,NaN,United Kingdom,True
7189,536997,22028,NaN,-20,2010-12-03 15:30:00,0.0,NaN,United Kingdom,True
7190,536998,85067,NaN,-6,2010-12-03 15:30:00,0.0,NaN,United Kingdom,True
...,...,...,...,...,...,...,...,...,...
535333,581210,23395,check,-26,2011-12-07 18:36:00,0.0,NaN,United Kingdom,True
535335,581212,22578,lost,-1050,2011-12-07 18:38:00,0.0,NaN,United Kingdom,True
535336,581213,22576,check,-30,2011-12-07 18:38:00,0.0,NaN,United Kingdom,True
536908,581226,23090,missing,-338,2011-12-08 09:56:00,0.0,NaN,United Kingdom,True


> **Из таблицы выше можно заметить что отрицательные _Quantity_ не отмеченные как возврат имеют _UnitPrice равное нулю_. Необходимо проверить корректно ли это утверждение для всех строк**

In [139]:
df1["UnitPrice"].unique()

array([0.])

In [140]:
df1["Description"].unique()

array([nan, '?', 'check', 'damages', 'faulty', 'Dotcom sales',
       'reverse 21/5/10 adjustment', 'mouldy, thrown away.', 'counted',
       'Given away', 'Dotcom', 'label mix up', 'samples/damages',
       'thrown away', 'incorrectly made-thrown away.', 'showroom', 'MIA',
       'Dotcom set', 'wrongly sold as sets', 'Amazon sold sets',
       'dotcom sold sets', 'wrongly sold sets', '? sold as sets?',
       '?sold as sets?', 'Thrown away.', 'damages/display',
       'damaged stock', 'broken', 'throw away', 'wrong barcode (22467)',
       'wrong barcode', 'barcode problem', '?lost',
       "thrown away-can't sell.", "thrown away-can't sell", 'damages?',
       're dotcom quick fix.', "Dotcom sold in 6's", 'sold in set?',
       'cracked', 'sold as 22467', 'Damaged',
       'mystery! Only ever imported 1800',
       'MERCHANT CHANDLER CREDIT ERROR, STO', 'POSSIBLE DAMAGES OR LOST?',
       'damaged', 'DAMAGED', 'Display', 'Missing', 'wrong code?',
       'wrong code', 'adjust', 'crush

> **Каждый отрицательный Quantity не отмеченный как возврат имеет UnitPrice равное нулю. По Description этих строк можно сделать вывод что эти транзакции не связаны с продажами компании и не будут влиять на итоговые метрики о прибольности, топ товарах, покупателях и тд. Поэтому при условии Quantity<0 & UnitPrice=0.0 данные можно считать некорректными и очистить от них таблицу.**

In [141]:
df.loc[df["UnitPrice"]==0.0]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsReturn
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom,False
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom,False
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom,False
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom,False
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom,False
...,...,...,...,...,...,...,...,...,...
536981,581234,72817,NaN,27,2011-12-08 10:33:00,0.0,NaN,United Kingdom,False
538504,581406,46000M,POLYESTER FILLER PAD 45x45cm,240,2011-12-08 13:58:00,0.0,NaN,United Kingdom,False
538505,581406,46000S,POLYESTER FILLER PAD 40x40cm,300,2011-12-08 13:58:00,0.0,NaN,United Kingdom,False
538554,581408,85175,NaN,20,2011-12-08 14:06:00,0.0,NaN,United Kingdom,False


> **Помимо ошибочно указанных как возврат "товаров", есть и товары с корректным _InvoiceNo, Description и Quantity_, но при этом с ценой в 0. В таких случаях есть возможность восстоновить цену за товар, через _StockCode_. Но в данном случае общее количество строк с UnitPrice=0 составляет 2515 строк, что есть лишь 0.5% от всех данных. Значит есть возможность в целом убрать все строки с отсутвующей ценой за товар.**

In [142]:
df_cleaned = df.drop(df.loc[df["UnitPrice"]==0.0].index)
df_cleaned

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsReturn
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,False
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,False
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False
...,...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France,False
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France,False
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France,False
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France,False


In [143]:
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 539394 entries, 0 to 541908
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    539394 non-null  object        
 1   StockCode    539394 non-null  object        
 2   Description  539394 non-null  object        
 3   Quantity     539394 non-null  int64         
 4   InvoiceDate  539394 non-null  datetime64[ns]
 5   UnitPrice    539394 non-null  float64       
 6   CustomerID   406789 non-null  float64       
 7   Country      539394 non-null  object        
 8   IsReturn     539394 non-null  bool          
dtypes: bool(1), datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 37.6+ MB


> **Таким образом ушли не только ошибочно указанные цены но также товары которые имеют отрицательный Quantity без соответсвующего StockCode маркера возврата.**

In [144]:
df_cleaned.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,539394.000000,539394,539394.000000,406789.000000
mean,9.845871,2011-07-04 16:40:48.702098944,4.632614,15287.795830
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:59:00,1.250000,13954.000000
50%,3.000000,2011-07-20 11:50:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:49:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,215.412253,NaN,96.984656,1713.573064


In [145]:
result=df_cleaned.groupby("StockCode")["Description"].unique().reset_index()
result = result[result["Description"].str.len() > 1]
result

,StockCode,Description
48,20622,"[VIPPASSPORT COVER , VIP PASSPORT COVER ]"
100,20725,"[LUNCH BAG RED RETROSPOT, LUNCH BAG RED SPOTTY]"
195,20914,"[SET/5 RED RETROSPOT LID GLASS BOWLS, SET/5 RE..."
294,21109,"[LARGE CAKE TOWEL, CHOCOLATE SPOTS, LARGE CAKE..."
297,21112,"[SWISS ROLL TOWEL, PINK SPOTS, SWISS ROLL TOW..."
...,...,...
3604,85184C,"[S/4 VALENTINE DECOUPAGE HEART BOX, SET 4 VALE..."
3606,85185B,"[PINK HORSE SOCK PUPPET, PINK HORSE SOCK PUPPE..."
3658,90014A,"[SILVER/MOP ORBIT NECKLACE, SILVER M.O.P. ORBI..."
3659,90014B,"[GOLD M PEARL ORBIT NECKLACE, GOLD M.O.P. ORB..."


> **По описанию можно заметить что его смысловая нагрузка одинакова для одного и того же товара, просто использовались сокращения, уточнения и изменения порядка слов. Необходимо проверить правдиво ли это утверждение для всех товаров имеющих несколько Description**

In [146]:
import re

def tokenize(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9 ]', '', text)
    return set(text.split())

In [147]:
def jaccard(a, b):
    set1, set2 = tokenize(a), tokenize(b)
    return len(set1 & set2) / len(set1 | set2)

In [148]:
codes = result["StockCode"]

for code in codes:
    descs = df_cleaned[df_cleaned["StockCode"] == code]["Description"].dropna().unique()

for i in range(len(descs)):
    for j in range(i+1, len(descs)):
        if jaccard(descs[i], descs[j])<0.8:
            print(descs[i], "|", descs[j], jaccard(descs[i], descs[j]))

SILVER/BLACK ORBIT NECKLACE | SILVER AND BLACK ORBIT NECKLACE 0.3333333333333333


> Из этого следует что товары имеющие несколько Description это все тот же товар, только при заполнения описания были изменения в написании, и можно привести все к одному наиболее повторяемому каноничному описанию. Такой подход может помочь полнеценно очистить дублирующиеся записи.

In [149]:
canonical = (
    df_cleaned.groupby("StockCode")["Description"]
    .agg(lambda x: x.value_counts().idxmax())
)

In [150]:
df_cleaned["Description"] = df_cleaned["StockCode"].map(canonical)
df_cleaned

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsReturn
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,False
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,False
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False
...,...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France,False
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France,False
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France,False
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France,False


> **После приведения Description к одному виду можно точно определить дубликаты и удалить их из таблицы**

In [151]:
df_cleaned.loc[
    df_cleaned["Description"] == "SILVER/BLACK ORBIT NECKLACE",
    "Description"
] = "SILVER AND BLACK ORBIT NECKLACE"

In [152]:
df_cleaned.loc[df.duplicated(keep='first')==False]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsReturn
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,False
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,False
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False
...,...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France,False
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France,False
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France,False
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France,False


In [153]:
df_cleaned=df_cleaned.loc[df.duplicated(keep='first')==False]
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 534131 entries, 0 to 541908
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    534131 non-null  object        
 1   StockCode    534131 non-null  object        
 2   Description  534131 non-null  object        
 3   Quantity     534131 non-null  int64         
 4   InvoiceDate  534131 non-null  datetime64[ns]
 5   UnitPrice    534131 non-null  float64       
 6   CustomerID   401564 non-null  float64       
 7   Country      534131 non-null  object        
 8   IsReturn     534131 non-null  bool          
dtypes: bool(1), datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 37.2+ MB


In [154]:
df_cleaned

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsReturn
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,False
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,False
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False
...,...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France,False
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France,False
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France,False
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France,False


In [155]:
df_cleaned["Description"].unique()

array(['WHITE HANGING HEART T-LIGHT HOLDER', 'WHITE METAL LANTERN',
       'CREAM CUPID HEARTS COAT HANGER', ...,
       'SET 10 CARDS SWIRLY XMAS TREE 17104', 'LETTER "U" BLING KEY RING',
       'PAPER CRAFT , LITTLE BIRDIE'], shape=(3812,), dtype=object)

In [156]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering

In [157]:
df_desc = (
    df_cleaned[["StockCode", "Description"]]
    .dropna()
    .drop_duplicates()
    .copy()
)

df_desc["is_upper"] = df_desc["Description"].str.fullmatch(r'[^a-zа-я]*', na=False)

In [158]:
df_desc[~df_desc["is_upper"]].sort_values(["StockCode", "Description"])

,StockCode,Description,is_upper
11417,18007,ESSENTIAL BALM 3.5g TIN IN ENVELOPE,False
35675,20954,*USB Office Mirror Ball,False
177542,20964,POLYESTER FILLER PAD 60x40cm,False
20749,21120,*Boombox Ipod Classic,False
2567,21594,Dr. Jam's Arouzer Stress Ball,False
13380,21595,Dad's Cab Electronic Meter,False
1961,21703,BAG 125g SWIRLY MARBLES,False
1962,21704,BAG 250g SWIRLY MARBLES,False
482,21705,BAG 500g SWIRLY MARBLES,False
287644,22016,Dotcomgiftshop Gift Voucher £100.00,False


In [159]:
bad_desc_mask = df_cleaned["Description"].str.contains(
    r"missing|dotcom|postage|amazon|voucher|manual|discount|samples|adjust|commission",
    case=False,
    na=False
)

In [160]:
df_cleaned.loc[bad_desc_mask, ["StockCode", "Description"]].drop_duplicates().sort_values("Description")

,StockCode,Description
14514,AMAZONFEE,AMAZON FEE
299982,B,Adjust bad debt
317508,CRUK,CRUK Commission
1814,DOT,DOTCOM POSTAGE
94777,22351,DOTCOMGIFTSHOP TEA TOWEL
141,D,Discount
112442,gift_0001_10,Dotcomgiftshop Gift Voucher £10.00
287644,22016,Dotcomgiftshop Gift Voucher £100.00
44794,gift_0001_20,Dotcomgiftshop Gift Voucher £20.00
44725,gift_0001_30,Dotcomgiftshop Gift Voucher £30.00


In [161]:
df_cleaned = df_cleaned.loc[~bad_desc_mask].copy()

In [172]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

# 1. Уникальные описания
products = (
    df_cleaned[["Description"]]
    .dropna()
    .drop_duplicates()
    .copy()
)

products["Description"] = products["Description"].astype(str).str.strip()
products = products[products["Description"] != ""].reset_index(drop=True)

# 2. Эмбеддинги
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

embeddings = model.encode(
    products["Description"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

kmeans = KMeans(
    n_clusters=40,
    random_state=42,
    n_init=10
)

products["cluster_id"] = kmeans.fit_predict(embeddings)

# 4. Сводка
cluster_summary = (
    products.groupby("cluster_id")["Description"]
    .agg(
        n_products="count",
        examples=lambda x: " | ".join(x.head(8))
    )
    .reset_index()
    .sort_values("n_products", ascending=False)
)

print(cluster_summary.head(40))

Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 4020.41it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|███████████████████████████████████████████████████████████████████████| 119/119 [00:05<00:00, 21.78it/s]


    cluster_id  n_products                                           examples
12          12         220  HAND WARMER UNION JACK | RED COAT RACK PARIS F...
28          28         206  YELLOW BREAKFAST CUP AND SAUCER | PINK BREAKFA...
24          24         188  SET 7 BABUSHKA NESTING BOXES | BOX OF 6 ASSORT...
19          19         181  JUMBO  BAG BAROQUE BLACK WHITE | JUMBO BAG CHA...
3            3         173  BALLOON ART MAKE YOUR OWN FLOWERS | IVORY GIAN...
7            7         166  ZINC WILLIE WINKIE  CANDLE STICK | HOMEMADE JA...
10          10         161  FELTCRAFT HAIRBAND RED AND BLUE | FELTCRAFT HA...
27          27         140  SPACEBOY LUNCH BOX | LUNCH BOX I LOVE LONDON |...
33          33         138  PANDA AND BUNNIES STICKER SHEET | MINI PAINT S...
0            0         133  VINTAGE HEADS AND TAILS CARD GAME | FANCY FONT...
21          21         126  CREAM CUPID HEARTS COAT HANGER | RED WOOLLY HO...
11          11         118  FELT EGG COSY CHICKEN | FELT EGG COS

In [173]:
cluster_summary.info()

<class 'pandas.core.frame.DataFrame'>
Index: 40 entries, 12 to 39
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   cluster_id  40 non-null     int32 
 1   n_products  40 non-null     int64 
 2   examples    40 non-null     object
dtypes: int32(1), int64(1), object(1)
memory usage: 1.1+ KB


In [174]:
top_clusters = (
    cluster_summary
    .sort_values("n_products", ascending=False)
    .head(40)["cluster_id"]
    .tolist()
)

In [179]:
all_clusters = (
    cluster_summary["cluster_id"]
    .tolist()
)

In [175]:
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("Key loaded:", api_key[:10] + "...")
else:
    print("Key not found")

Key loaded: sk-proj-N8...


In [176]:
from openai import OpenAI

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [177]:
def label_cluster_with_openai(product_examples, model_name="gpt-4o-mini"):
    examples_text = "\n".join(
        f"- {x}" for x in product_examples
        if pd.notna(x) and str(x).strip()
    )

    prompt = f"""
You are given a cluster of similar e-commerce product names.

Products:
{examples_text}

Task:
Give one short product category name for this cluster.

Rules:
- 2 to 4 words
- no explanation
- return only the category name
- use broad but meaningful retail category wording
- do NOT use vague names like "Home Accessories", "Items", "Miscellaneous", "Home & Garden"
- focus on the actual product type, not color, style, or minor variation

Good examples:
Candle Holders
Lunch Boxes
Greeting Cards
Drop Earrings
Storage Bags
Wall Mirrors
Kids Toys
Home Decor
Kitchen Storage
Christmas Decor
"""

    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": "You label retail product clusters."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=20,
        temperature=0
    )

    return response.choices[0].message.content.strip()

In [180]:
cluster_labels = []

for cluster_id in all_clusters:
    examples = (
        products.loc[products["cluster_id"] == cluster_id, "Description"]
        .dropna()
        .astype(str)
        .head(10)
        .tolist()
    )

    print(f"Processing cluster {cluster_id}...")
    category_label = label_cluster_with_openai(examples)

    cluster_labels.append({
        "cluster_id": cluster_id,
        "category": category_label
    })

cluster_labels_df = pd.DataFrame(cluster_labels)
cluster_labels_df

Processing cluster 12...
Processing cluster 28...
Processing cluster 24...
Processing cluster 19...
Processing cluster 3...
Processing cluster 7...
Processing cluster 10...
Processing cluster 27...
Processing cluster 33...
Processing cluster 0...
Processing cluster 21...
Processing cluster 11...
Processing cluster 16...
Processing cluster 1...
Processing cluster 25...
Processing cluster 23...
Processing cluster 8...
Processing cluster 5...
Processing cluster 37...
Processing cluster 14...
Processing cluster 30...
Processing cluster 35...
Processing cluster 17...
Processing cluster 31...
Processing cluster 4...
Processing cluster 20...
Processing cluster 18...
Processing cluster 26...
Processing cluster 29...
Processing cluster 9...
Processing cluster 13...
Processing cluster 6...
Processing cluster 22...
Processing cluster 36...
Processing cluster 32...
Processing cluster 15...
Processing cluster 38...
Processing cluster 34...
Processing cluster 2...
Processing cluster 39...


,cluster_id,category
0,12,Decorative Accessories
1,28,Tableware and Kitchenware
2,24,Party Supplies
3,19,Bags and Purses
4,3,Garden Decor
5,7,Scented Candles
6,10,Hair Accessories and Jewelry
7,27,Lunch Boxes and Accessories
8,33,Craft Supplies
9,0,Playing Cards and Stationery


In [181]:
products = products.drop(columns=["category"], errors="ignore")

products = products.merge(
    cluster_labels_df,
    on="cluster_id",
    how="left"
)

products.head(20)

,Description,cluster_id,category
0,WHITE HANGING HEART T-LIGHT HOLDER,35,T-Light Holders
1,WHITE METAL LANTERN,5,Decorative Lighting
2,CREAM CUPID HEARTS COAT HANGER,21,Heart Decor
3,KNITTED UNION FLAG HOT WATER BOTTLE,36,Hot Water Bottles
4,RED WOOLLY HOTTIE WHITE HEART.,21,Heart Decor
5,SET 7 BABUSHKA NESTING BOXES,24,Party Supplies
6,GLASS STAR FROSTED T-LIGHT HOLDER,35,T-Light Holders
7,HAND WARMER UNION JACK,12,Decorative Accessories
8,HAND WARMER RED RETROSPOT,30,Kitchen and Dining Accessories
9,ASSORTED COLOUR BIRD ORNAMENT,37,Bird Decor


In [182]:
category_map = {
    "Decorative Accessories": "Home Decor",
    "Garden Decor": "Home Decor",
    "Heart Decor": "Home Decor",
    "Heart-Shaped Decor": "Home Decor",
    "Rose-Themed Home Decor": "Home Decor",
    "Bird Decor": "Home Decor",
    "Glass Decor Items": "Home Decor",
    "Zinc Decorations": "Home Decor",
    "Decorative Lighting": "Home Decor",
    "Metal Signs": "Home Decor",
    "House Number Tiles": "Home Decor",
    "Kids Room Decor": "Home Decor",

    "Tableware and Kitchenware": "Kitchen & Dining",
    "Kitchen and Dining Accessories": "Kitchen & Dining",
    "Lunch Boxes and Accessories": "Kitchen & Dining",
    "Coffee Mugs": "Kitchen & Dining",
    "Tissue Boxes": "Kitchen & Dining",

    "Scented Candles": "Candles & Holders",
    "T-Light Holders": "Candles & Holders",

    "Hair Accessories and Jewelry": "Jewelry",
    "Drop Earrings": "Jewelry",

    "Bags and Purses": "Bags & Storage",
    "Drawer Hardware & Storage": "Bags & Storage",
    "Keepsake Boxes": "Bags & Storage",

    "Gift Wrap": "Gift & Party Supplies",
    "Party Supplies": "Gift & Party Supplies",

    "Notebooks and Journals": "Stationery",
    "Playing Cards and Stationery": "Stationery",

    "Craft Supplies": "Kids & Crafts",
    "Kids Dolls and Accessories": "Kids & Crafts",

    "Christmas Decorations": "Seasonal Decor",
    "Christmas Ribbons": "Seasonal Decor",
    "Easter Decorations": "Seasonal Decor",

    "Alarm Clocks": "Home Accessories",
    "Hot Water Bottles": "Home Accessories",
    "Photo Frames & Mirrors": "Home Accessories",

    "Polkadot Accessories": "Accessories",
    "Paisley Print Products": "Accessories",
    "Key Rings": "Accessories"
}

In [183]:
products["category_business"] = products["category"].map(category_map)
products["category_business"] = products["category_business"].fillna(products["category"])

products[["Description", "cluster_id", "category", "category_business"]].head(20)

,Description,cluster_id,category,category_business
0,WHITE HANGING HEART T-LIGHT HOLDER,35,T-Light Holders,Candles & Holders
1,WHITE METAL LANTERN,5,Decorative Lighting,Home Decor
2,CREAM CUPID HEARTS COAT HANGER,21,Heart Decor,Home Decor
3,KNITTED UNION FLAG HOT WATER BOTTLE,36,Hot Water Bottles,Home Accessories
4,RED WOOLLY HOTTIE WHITE HEART.,21,Heart Decor,Home Decor
5,SET 7 BABUSHKA NESTING BOXES,24,Party Supplies,Gift & Party Supplies
6,GLASS STAR FROSTED T-LIGHT HOLDER,35,T-Light Holders,Candles & Holders
7,HAND WARMER UNION JACK,12,Decorative Accessories,Home Decor
8,HAND WARMER RED RETROSPOT,30,Kitchen and Dining Accessories,Kitchen & Dining
9,ASSORTED COLOUR BIRD ORNAMENT,37,Bird Decor,Home Decor


In [186]:
df_cleaned["Description"] = df_cleaned["Description"].astype(str).str.strip()
products["Description"] = products["Description"].astype(str).str.strip()

df_cleaned = df_cleaned.drop(columns=["cluster_id", "category", "category_business"], errors="ignore")

df_cleaned = df_cleaned.merge(
    products[["Description", "category_business"]],
    on="Description",
    how="left"
)

In [187]:
df_cleaned.head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsReturn,category_business
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,False,Candles & Holders
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,Home Decor
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,False,Home Decor
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,Home Accessories
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,Home Decor
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.0,United Kingdom,False,Gift & Party Supplies
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.0,United Kingdom,False,Candles & Holders
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom,False,Home Decor
8,536366,22632,HAND WARMER RED RETROSPOT,6,2010-12-01 08:28:00,1.85,17850.0,United Kingdom,False,Kitchen & Dining
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047.0,United Kingdom,False,Home Decor


In [188]:
df_cleaned["category_business"].isna().sum()

np.int64(0)

In [189]:
df_cleaned = df_cleaned.copy()
df_cleaned["Revenue"]=df_cleaned["Quantity"]*df_cleaned["UnitPrice"]

In [190]:
df_cleaned["Month"]=df_cleaned['InvoiceDate'].dt.month

In [191]:
df_cleaned["Year"]=df_cleaned['InvoiceDate'].dt.year

In [192]:
df_cleaned["YearMonth"]=df_cleaned['InvoiceDate'].dt.to_period("M").astype(str)

In [193]:
df_cleaned = df_cleaned.rename(columns={
    "InvoiceNo": "invoice_no",
    "StockCode": "stock_code",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "UnitPrice": "unit_price",
    "CustomerID": "customer_id",
    "Country": "country",
    "IsReturn": "is_return",
    "category_business": "category",
    "Revenue": "revenue",
    "Year": "year",
    "Month": "month",
    "YearMonth": "year_month"
})
df_cleaned

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,is_return,category,revenue,month,year,year_month
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,False,Candles & Holders,15.30,12,2010,2010-12
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,Home Decor,20.34,12,2010,2010-12
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,False,Home Decor,22.00,12,2010,2010-12
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,Home Accessories,20.34,12,2010,2010-12
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,Home Decor,20.34,12,2010,2010-12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
535356,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France,False,Gift & Party Supplies,10.20,12,2011,2011-12
535357,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France,False,Kids & Crafts,12.60,12,2011,2011-12
535358,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France,False,Kids & Crafts,16.60,12,2011,2011-12
535359,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France,False,Kitchen & Dining,16.60,12,2011,2011-12


In [195]:
df_cleaned.to_csv("online_retail_clean.csv", index=False)